# 02 — Combine daily files into per-sensor Parquet

Match each study sensor to its folder on disk (by the `SLxxx` code, which appears in both the summary name and the folder name), stack all daily CSVs into one Parquet per sensor in `data/processed/`, and write `sensor_inventory.csv` (the authoritative folder list the incremental loader later depends on).

**This is the full-rebuild path.** For day-to-day refreshes once this has run, use `python -m src.loader` instead, which only reads new/changed days.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # so `from src...` works from notebooks/
from src.config import *
import pandas as pd
import re, time

study = pd.read_csv(OUT_TABLES / "study_sensors.csv")
print(f"Study sensors to combine: {len(study)}")
study.head()

Study sensors to combine: 48


,sensor_id,sensor_name,location_type,lat,lon
0,15,SL001 Sunnyview Terrace,outdoor,53.774956,-1.565842
1,27,SL003 - Corn Exchange Cabinet,outdoor,53.796494,-1.540428
2,26,SL005 Kirkstall Valley Primary,outdoor,53.808520,-1.586729
3,47,SL006 - Primrose Hill Primary,outdoor,53.801888,-1.667109
4,45,SL007 Ninelands Primary,outdoor,53.789272,-1.377445


### Match each study sensor to its folder on disk
Match by the `SLxxx` code rather than the full name, because punctuation differs between the summary (`SL001 Sunnyview Terrace`) and the folder (`SL001_Sunnyview_Terrace`).

In [2]:
def is_daily(name):
    return re.fullmatch(r"\d{4}-\d{2}-\d{2}\.csv", name) is not None

disk_folders = []
for item in sorted(DATA_DIR.iterdir()):
    if item.is_dir() and any(is_daily(p.name) for p in item.rglob("*.csv")):
        disk_folders.append(item.name)
print(f"Folders on disk with daily data: {len(disk_folders)}")

def sl_code(text):
    m = re.search(r"SL[_ ]?0*(\d+)", str(text).upper())
    return f"SL{int(m.group(1)):03d}" if m else None

study["code"] = study["sensor_name"].apply(sl_code)
folder_by_code = {}
for f in disk_folders:
    c = sl_code(f)
    if c:
        folder_by_code[c] = f

study["folder"] = study["code"].map(folder_by_code)
matched   = study.dropna(subset=["folder"])
unmatched = study[study["folder"].isna()]
print(f"Matched to a folder: {len(matched)}")
print(f"NOT matched (no folder found): {len(unmatched)}")
if len(unmatched):
    print(unmatched[["sensor_name", "code"]].to_string(index=False))

Folders on disk with daily data: 72
Matched to a folder: 47
NOT matched (no folder found): 1
    sensor_name  code
SL71 - Kentmere SL071


### Combine function
Stack all daily CSVs for one sensor, parse the timestamp, return the frame plus per-sensor stats.

In [3]:
def combine_one_sensor(folder_name):
    """Stack all daily CSVs for one sensor. Returns (dataframe, stats dict)."""
    folder = DATA_DIR / folder_name
    files  = sorted(p for p in folder.rglob("*.csv") if is_daily(p.name))

    frames, empty_days, bad_files = [], 0, 0
    for f in files:
        try:
            d = pd.read_csv(f)
            if len(d) == 0:
                empty_days += 1
            else:
                frames.append(d)
        except Exception:
            bad_files += 1

    if not frames:
        return pd.DataFrame(), {
            "folder": folder_name, "rows": 0, "days_with_data": 0,
            "empty_days": empty_days, "bad_files": bad_files,
            "start": None, "end": None,
        }

    df = pd.concat(frames, ignore_index=True)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], format="ISO8601", errors="coerce")
    df = df.sort_values(DATE_COL).reset_index(drop=True)
    df.insert(0, "sensor", folder_name)

    return df, {
        "folder": folder_name, "rows": len(df),
        "days_with_data": df[DATE_COL].dt.date.nunique(),
        "empty_days": empty_days, "bad_files": bad_files,
        "start": df[DATE_COL].min(), "end": df[DATE_COL].max(),
    }

### Run the combine over all matched sensors
First run is slow (reads every daily file once). Writes one Parquet per sensor.

In [4]:
inventory = []
t0 = time.time()
folders_to_do = matched["folder"].tolist()

for i, fname in enumerate(folders_to_do, 1):
    print(f"[{i}/{len(folders_to_do)}] {fname} ...", end=" ")
    df, stats = combine_one_sensor(fname)
    if df.empty:
        print("no data")
    else:
        df.to_parquet(PROCESSED / f"{fname}.parquet", index=False)
        print(f"{stats['rows']:,} rows  ({stats['days_with_data']} days, "
              f"{stats['empty_days']} empty)")
    inventory.append(stats)

print(f"\nDone in {time.time()-t0:.0f}s")

[1/47] SL001_Sunnyview_Terrace ... 1,038,078 rows  (1445 days, 287 empty)
[2/47] SL003_Corn_Exchange_Cabinet ... 575,937 rows  (829 days, 903 empty)
[3/47] SL005_Kirkstall_Valley_Primary ... 523,797 rows  (762 days, 970 empty)
[4/47] SL006 ... 1,035,653 rows  (1455 days, 274 empty)
[5/47] SL007_Ninelands_Primary ... 1,025,036 rows  (1448 days, 281 empty)
[6/47] SL008_Westgate_Primary ... 771,113 rows  (1109 days, 620 empty)
[7/47] SL009 ... 168,047 rows  (254 days, 1475 empty)
[8/47] SL011 ... 807,368 rows  (1437 days, 295 empty)
[9/47] SL012_Morley ... 934,514 rows  (1304 days, 425 empty)
[10/47] SL015_-_Garforth_Main_Street ... 417,638 rows  (590 days, 1142 empty)
[11/47] SL016_Scholes_Main_Street ... 720,567 rows  (1324 days, 408 empty)
[12/47] SL_017_-_Boston_Spa_Town_Centre ... 470,881 rows  (939 days, 793 empty)
[13/47] SL018_East_Ardsley_A650 ... 632,321 rows  (1168 days, 564 empty)
[14/47] SL019_Cornwall_Crescent_Rothwell ... 735,371 rows  (1184 days, 548 empty)
[15/47] SL020 .

### Build the inventory table
This `sensor_inventory.csv` (with its `folder` column) is what `src/loader.py` reads to know which folders to refresh.

In [5]:
inv = pd.DataFrame(inventory)
inv = inv.merge(matched[["folder", "sensor_name", "lat", "lon"]],
                on="folder", how="left")

inv["span_days"]    = (pd.to_datetime(inv["end"]) - pd.to_datetime(inv["start"])).dt.days + 1
inv["coverage_pct"] = (inv["days_with_data"] / inv["span_days"] * 100).round(1)

cols = ["sensor_name", "folder", "rows", "days_with_data", "empty_days",
        "span_days", "coverage_pct", "start", "end", "bad_files"]
inv = inv[cols].sort_values("sensor_name")
inv.to_csv(OUT_TABLES / "sensor_inventory.csv", index=False)
print("Saved -> outputs/tables/sensor_inventory.csv")
inv

Saved -> outputs/tables/sensor_inventory.csv


,sensor_name,folder,rows,days_with_data,empty_days,span_days,coverage_pct,start,end,bad_files
0,SL001 Sunnyview Terrace,SL001_Sunnyview_Terrace,1038078,1445,287,1446,99.9,2022-03-07 18:37:08+00:00,2026-02-19 18:57:20+00:00,0
1,SL003 - Corn Exchange Cabinet,SL003_Corn_Exchange_Cabinet,575937,829,903,1081,76.7,2022-03-09 14:45:42+00:00,2025-02-22 00:01:37+00:00,0
2,SL005 Kirkstall Valley Primary,SL005_Kirkstall_Valley_Primary,523797,762,970,1313,58.0,2022-07-05 08:58:00+00:00,2026-02-06 11:27:41+00:00,0
3,SL006 - Primrose Hill Primary,SL006,1035653,1455,274,1580,92.1,2022-03-09 15:09:03+00:00,2026-07-05 23:59:23+00:00,0
4,SL007 Ninelands Primary,SL007_Ninelands_Primary,1025036,1448,281,1580,91.6,2022-03-09 15:15:38+00:00,2026-07-05 23:58:22+00:00,0
5,SL008 Westgate Primary,SL008_Westgate_Primary,771113,1109,620,1580,70.2,2022-03-09 15:26:26+00:00,2026-07-05 23:58:46+00:00,0
6,SL009 - Roundhay Golf Club,SL009,168047,254,1475,834,30.5,2022-03-09 15:33:26+00:00,2024-06-20 09:27:33+00:00,0
7,"SL011 - West End Grove, Horsforth",SL011,807368,1437,295,1580,90.9,2022-03-09 15:47:23+00:00,2026-07-05 23:58:18+00:00,0
8,"SL012 - Fountain Street, Morley",SL012_Morley,934514,1304,425,1496,87.2,2022-03-09 15:51:22+00:00,2026-04-13 05:45:31+00:00,0
9,SL015 - Garforth Main Street,SL015_-_Garforth_Main_Street,417638,590,1142,809,72.9,2022-03-22 15:59:02+00:00,2024-06-08 00:34:40+00:00,0
